# Quantum Variational Classifier — Iris Dataset Extension

This notebook extends Qiskit's `VQC` to multiclass classification on the Iris dataset using a two-layer decision strategy: One-vs-Rest (OvR) binary VQCs as the first layer, and One-vs-One (OvO) tiebreaker classifiers as the second layer. Classical SVC baselines are included for comparison.

---

## 1. Installation

In [2]:
!pip install -q qiskit pylatexenc qiskit_algorithms qiskit_machine_learning tinydb dill

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.5 MB/s eta 0:00:00


## 2. Imports

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import time
import json
import base64
import hashlib
import os
from itertools import combinations
from dataclasses import dataclass, field, asdict, replace
from typing import Optional, List

from sklearn.datasets import load_iris
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC

from qiskit.circuit.library import zz_feature_map, z_feature_map, real_amplitudes, efficient_su2
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.optimizers import COBYLA, SLSQP
from qiskit_machine_learning.primitives import QMLSampler
from qiskit_machine_learning.utils import algorithm_globals
from qiskit_machine_learning.algorithms.classifiers import VQC
from IPython.display import clear_output

import dill
from tinydb import TinyDB, Query

## 3. Google Drive & Database setup

Runs on Colab (mounts Drive) and falls back to a local path when running outside Colab.

In [4]:
try:
    from google.colab import drive, auth
    auth.authenticate_user()
    drive.mount('/content/drive', force_remount=True)
    DB_PATH = "/content/drive/MyDrive/TinyDb/qml_qvc_db.json"
    os.makedirs("/content/drive/MyDrive/TinyDb", exist_ok=True)
    print("Running on Colab — DB on Drive.")
except ImportError:
    DB_PATH = "./qml_qvc_db.json"
    print("Running locally — DB in current directory.")

db           = TinyDB(DB_PATH)
models_table = db.table("qvc_iris_exact")
print(f"DB path: {DB_PATH}")

Mounted at /content/drive
Running on Colab — DB on Drive.
DB path: /content/drive/MyDrive/TinyDb/qml_qvc_db.json


---
## 4. Data classes

`ModelConfig` captures every hyperparameter. `ModelResult` stores training metrics. `ModelRecord` ties them together and produces a deterministic MD5 id from the config so duplicate experiments are never saved twice.

In [5]:
@dataclass
class ModelConfig:
    # Dataset
    data_set:            str   = "iris"
    scale_features:      bool  = True
    min_scale_features:  float = 0
    max_scale_features:  float = np.pi

    # Train/test split
    train_size:          float = 0.8
    random_seed:         int   = 123
    shuffle:             bool  = True
    binary:              bool  = False
    info:                str   = "---"

    # PCA (unused in current experiments, reserved for future use)
    pca_apply:           bool  = False
    pca_n_components:    int   = 2

    # Feature map
    feature_map_fn:      str   = "zz_feature_map"
    feature_map_rep:     int   = 1

    # Ansatz
    ansatz_fn:               str = "real_amplitudes"
    ansatz_fn_rep:           int = 3
    ansatz_fn_entanglement:  str = "full"  # 'full', 'linear', 'reverse_linear', 'circular', 'sca'

    # Optimizer
    optimizer_type:      str = "cobyla"
    optimizer_maxiter:   int = 100

    # Sampler
    algorithm_global_random_seed: int           = 42
    sampler_type:                 str           = "QMLSampler"
    sampler_seed:                 Optional[int] = None

    # Plotting
    plot_graph:          bool = False


@dataclass
class ModelResult:
    objective_func_vals: List[float]     = field(default_factory=list)
    train_acc_vals:      List[float]     = field(default_factory=list)
    test_acc_vals:       List[float]     = field(default_factory=list)
    time_elapsed:        Optional[float] = None
    train_score:         Optional[float] = None
    test_score:          Optional[float] = None
    model_dill:          Optional[str]   = None


@dataclass
class ModelRecord:
    config: ModelConfig
    result: ModelResult = field(default_factory=ModelResult)
    id:     str         = field(init=False)

    def __post_init__(self):
        self.id = self._hash_config()

    def _hash_config(self) -> str:
        config_str = json.dumps(asdict(self.config), sort_keys=True)
        return hashlib.md5(config_str.encode()).hexdigest()

## 5. Database helpers

In [6]:
def save_record(record: ModelRecord) -> None:
    """Insert a new record. Skips silently if the config hash already exists."""
    M = Query()
    if models_table.contains(M.id == record.id):
        print(f"Record '{record.id}' already exists, skipping.")
        return
    models_table.insert(asdict(record))
    print(f"Record '{record.id}' saved.")


def update_record(record_id: str, config: ModelConfig = None, result: ModelResult = None) -> None:
    """Merge updated config and/or result fields into an existing record."""
    M = Query()
    existing = models_table.get(M.id == record_id)
    if existing is None:
        raise KeyError(f"Record '{record_id}' not found.")
    updated = {}
    if config:
        updated["config"] = {**existing["config"], **asdict(config)}
    if result:
        updated["result"] = {**existing["result"], **asdict(result)}
    models_table.update(updated, M.id == record_id)
    print(f"Record '{record_id}' updated.")


def load_record(record_id: str) -> ModelRecord:
    """Load a single record by its MD5 id."""
    M = Query()
    raw = models_table.get(M.id == record_id)
    if raw is None:
        raise KeyError(f"Record '{record_id}' not found.")
    return ModelRecord(config=ModelConfig(**raw["config"]), result=ModelResult(**raw["result"]))


def query_records(predicate) -> List[ModelRecord]:
    """
    Retrieve all records matching a predicate applied to a ModelRecord instance.

    Examples
    --------
    query_records(lambda r: r.config.ansatz_fn_rep > 2)
    query_records(lambda r: r.result.test_score is not None and r.result.test_score > 0.9)
    query_records(lambda r: r.config.optimizer_type == "cobyla")
    """
    def _wrap(raw):
        r = ModelRecord(config=ModelConfig(**raw["config"]), result=ModelResult(**raw["result"]))
        return predicate(r)
    return [
        ModelRecord(config=ModelConfig(**raw["config"]), result=ModelResult(**raw["result"]))
        for raw in models_table.search(_wrap)
    ]

## 6. Component registry & builders

In [7]:
OPTIMIZER_MAP = {
    "cobyla": COBYLA,
    "slsqp":  SLSQP,
}

FEATURE_MAP = {
    "z_feature_map":  z_feature_map,
    "zz_feature_map": zz_feature_map,
}

ANSATZ_MAP = {
    "real_amplitudes": real_amplitudes,
    "efficient_su2":   efficient_su2,
}

SAMPLER_MAP = {
    "state_vector_sampler": StatevectorSampler,
    "QMLSampler":           QMLSampler,
}

DATA_SET_MAP = {
    "iris": load_iris,
}

In [8]:
def load_dataset(config: ModelConfig):
    """
    Load and scale the dataset, then split into train/test sets.

    Returns
    -------
    train_features, test_features, train_labels, test_labels, num_features
    """
    data_set        = DATA_SET_MAP[config.data_set]()
    features        = data_set.data
    labels          = data_set.target
    features_scaled = MinMaxScaler(
        feature_range=(config.min_scale_features, config.max_scale_features)
    ).fit_transform(features)

    X = features_scaled if config.scale_features else features
    train_features, test_features, train_labels, test_labels = train_test_split(
        X, labels,
        train_size=config.train_size,
        random_state=config.random_seed,
        shuffle=config.shuffle,
    )
    return train_features, test_features, train_labels, test_labels, train_features.shape[1]


def build_components(config: ModelConfig, num_features: int):
    """
    Instantiate feature map, ansatz, optimizer, and sampler from config.

    Returns
    -------
    feature_map, ansatz, optimizer, sampler
    """
    algorithm_globals.random_seed = config.algorithm_global_random_seed
    feature_map = FEATURE_MAP[config.feature_map_fn](feature_dimension=num_features, reps=config.feature_map_rep)
    ansatz      = ANSATZ_MAP[config.ansatz_fn](num_qubits=num_features, reps=config.ansatz_fn_rep, entanglement=config.ansatz_fn_entanglement)
    optimizer   = OPTIMIZER_MAP[config.optimizer_type](maxiter=config.optimizer_maxiter)
    sampler     = SAMPLER_MAP[config.sampler_type](seed=config.sampler_seed)
    return feature_map, ansatz, optimizer, sampler

## 7. Training

In [9]:
def train_and_save(
    sampler,
    feature_map,
    ansatz,
    optimizer,
    train_features,
    train_labels,
    test_features,
    test_labels,
    config: ModelConfig,
    initial_point=None,
):
    """
    Build, train, and persist a VQC model.

    Parameters
    ----------
    initial_point : array-like, optional
        Initial parameter vector. Defaults to all-0.5 if not provided.

    Returns
    -------
    vqc       : fitted VQC instance
    result    : ModelResult with scores and training curves
    record    : ModelRecord persisted to TinyDB
    """
    has_test = len(test_features) > 0

    if initial_point is None:
        initial_point = np.full(ansatz.num_parameters, 0.5)

    # --- tracking lists captured by callback ---
    objective_func_vals = []
    train_acc_vals      = []
    test_acc_vals       = []

    # --- build VQC first so callback closure can reference it ---
    vqc = VQC(
        sampler=sampler,
        feature_map=feature_map,
        ansatz=ansatz,
        optimizer=optimizer,
        initial_point=initial_point,
    )

    def callback_graph(weights, obj_func_eval):
        objective_func_vals.append(obj_func_eval)
        train_output = vqc._neural_network.forward(train_features, weights)
        train_acc    = accuracy_score(train_labels, np.argmax(train_output, axis=1))
        train_acc_vals.append(train_acc)

        if has_test:
            test_output = vqc._neural_network.forward(test_features, weights)
            test_acc    = accuracy_score(test_labels, np.argmax(test_output, axis=1))
            test_acc_vals.append(test_acc)

        if config.plot_graph:
            clear_output(wait=True)
            fig, ax1 = plt.subplots(figsize=(12, 5))
            ax1.set_xlabel("Iteration")
            ax1.set_ylabel("Loss", color="tab:blue")
            ax1.plot(objective_func_vals, color="tab:blue", label="Loss")
            ax1.tick_params(axis="y", labelcolor="tab:blue")
            ax2 = ax1.twinx()
            ax2.set_ylabel("Accuracy", color="tab:gray")
            ax2.plot(train_acc_vals, color="tab:orange", label="Train acc")
            if has_test:
                ax2.plot(test_acc_vals, color="tab:green", label="Test acc")
            ax2.set_ylim(0, 1.05)
            ax2.tick_params(axis="y", labelcolor="tab:gray")
            lines1, labels1 = ax1.get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
            plt.title("Loss and Accuracy vs Iteration")
            plt.tight_layout()
            plt.show()

    vqc.callback = callback_graph

    # --- train ---
    start = time.time()
    vqc.fit(train_features, train_labels)
    elapsed = time.time() - start

    train_score = vqc.score(train_features, train_labels)
    test_score  = vqc.score(test_features, test_labels) if has_test else 0.0

    result = ModelResult(
        time_elapsed=elapsed,
        train_score=train_score,
        test_score=test_score,
        model_dill=base64.b64encode(dill.dumps(vqc)).decode("utf-8"),
        train_acc_vals=train_acc_vals,
        test_acc_vals=test_acc_vals,
        objective_func_vals=objective_func_vals,
    )

    record = ModelRecord(config=config)
    save_record(record)
    update_record(record.id, config=config, result=result)

    all_records = query_records(lambda r: True)
    print(f"[DB] Total records: {len(all_records)}")
    print(f"Train: {train_score:.4f} | Test: {test_score:.4f} | Time: {elapsed:.1f}s")
    return vqc, result, record

## 8. Binary dataset helpers

In [10]:
def load_dataset_binary(config: ModelConfig) -> list:
    """
    Naive OvR binary split — train/test split happens before binarising labels.
    Each binary classifier sees an imbalanced dataset (1/3 positive, 2/3 negative).

    Returns
    -------
    list of (train_features, test_features, train_labels, test_labels) — one per class
    """
    data_set        = DATA_SET_MAP[config.data_set]()
    features_scaled = MinMaxScaler(
        feature_range=(config.min_scale_features, config.max_scale_features)
    ).fit_transform(data_set.data)

    X      = features_scaled if config.scale_features else data_set.data
    labels = data_set.target
    binary_dataset = []

    for i in range(len(data_set.target_names)):
        tr_f, te_f, tr_l, te_l = train_test_split(
            X, labels,
            train_size=config.train_size,
            random_state=config.random_seed,
            shuffle=config.shuffle,
        )
        binary_dataset.append((tr_f, te_f,
                                np.where(tr_l == i, 1, 0),
                                np.where(te_l == i, 1, 0)))
    return binary_dataset

In [11]:
def load_data_set_binary_balanced(
    X: np.ndarray,
    y: np.ndarray,
    random_state: int = 123,
) -> list:
    """
    Creates one balanced binary dataset per unique class (OvR strategy).

    For each target class:
      - positive (1): all samples of that class
      - negative (0): equal samples drawn from each other class (undersampled)
      - result is a 50/50 balanced dataset

    The test set is intentionally never passed here — balancing is applied
    only to training data to preserve test set integrity.

    Parameters
    ----------
    X            : feature matrix (n_samples, n_features)
    y            : multiclass label vector (n_samples,)
    random_state : for reproducibility

    Returns
    -------
    list of (X_balanced, y_binary) tuples — one per unique class
    """
    unique_classes = np.unique(y)
    datasets       = []

    for target_class in unique_classes:
        y_binary   = (y == target_class).astype(int)
        X_positive = X[y_binary == 1]
        y_positive = y_binary[y_binary == 1]
        n_positive = len(X_positive)

        negative_classes = [c for c in unique_classes if c != target_class]
        n_per_neg_class  = n_positive // len(negative_classes)

        X_neg_parts, y_neg_parts = [], []
        for neg_class in negative_classes:
            X_neg = X[y == neg_class]
            y_neg = y_binary[y == neg_class]
            X_down, y_down = resample(
                X_neg, y_neg,
                replace=False,
                n_samples=min(n_per_neg_class, len(X_neg)),
                random_state=random_state,
            )
            X_neg_parts.append(X_down)
            y_neg_parts.append(y_down)

        X_bal = np.vstack([X_positive, *X_neg_parts])
        y_bal = np.hstack([y_positive, *y_neg_parts])
        idx   = np.random.default_rng(random_state).permutation(len(y_bal))
        X_bal, y_bal = X_bal[idx], y_bal[idx]

        print(f"Class {target_class} vs Rest | pos: {sum(y_bal==1)}  neg: {sum(y_bal==0)}  total: {len(y_bal)}")
        datasets.append((X_bal, y_bal))

    return datasets


def load_dataset_binary_balanced_with_test(config: ModelConfig):
    """
    Balanced OvR split that preserves test set integrity.

    Strategy
    --------
    1. Train/test split is performed once on the original multiclass data.
    2. Balanced binary training sets are created per class (undersampling negatives).
    3. The test set is binarised per class but never resampled.

    Returns
    -------
    binary_dataset : list of (X_train_bal, X_test, y_train_bin, y_test_bin) per class
    X_train, X_test, y_train, y_test : original multiclass splits (needed for OvO tiebreakers)
    """
    data_set        = DATA_SET_MAP[config.data_set]()
    features_scaled = MinMaxScaler(
        feature_range=(config.min_scale_features, config.max_scale_features)
    ).fit_transform(data_set.data)

    X = features_scaled if config.scale_features else data_set.data
    X_train, X_test, y_train, y_test = train_test_split(
        X, data_set.target,
        train_size=config.train_size,
        random_state=config.random_seed,
        shuffle=config.shuffle,
    )

    train_datasets = load_data_set_binary_balanced(X_train, y_train, random_state=config.random_seed)

    binary_dataset = []
    for target_class, (X_train_bal, y_train_bin) in enumerate(train_datasets):
        y_test_bin = (y_test == target_class).astype(int)
        binary_dataset.append((X_train_bal, X_test, y_train_bin, y_test_bin))

    return binary_dataset, X_train, X_test, y_train, y_test

## 9. Decode helpers

`decode_ovr_naive` applies majority vote with random tie-breaking — no second layer.

`decode_ovr_with_ovo_tiebreaker` is the full two-layer decoder.

In [12]:
def decode_ovr_naive(models: list, X_test: np.ndarray, y_test: np.ndarray) -> float:
    """
    Decode OvR binary predictions using majority vote with random tie-breaking.
    No second decision layer.

    Parameters
    ----------
    models : list of fitted binary classifiers, one per class
    X_test : shared test feature matrix
    y_test : true multiclass labels

    Returns
    -------
    accuracy : float
    """
    n_classes  = len(models)
    n_samples  = len(y_test)
    pred_matrix = np.empty((n_classes, n_samples))

    for i, model in enumerate(models):
        pred_matrix[i] = model.predict(X_test)

    decoded  = np.empty(n_samples)
    n_bad    = 0

    for i, row in enumerate(pred_matrix.T):
        winners = np.where(row == 1)[0]
        if len(winners) == 1:
            decoded[i] = winners[0]
        elif len(winners) > 1:
            decoded[i] = random.choice(winners)
            n_bad += 1
        else:
            decoded[i] = random.choice(range(n_classes))
            n_bad += 1

    accuracy = sum(y_test == decoded) / n_samples
    print(f"Bad pred ratio : {n_bad}/{n_samples} ({n_bad/n_samples:.1%})")
    print(f"Accuracy       : {accuracy:.4f}")
    return accuracy

In [13]:
def decode_ovr_with_ovo_tiebreaker(
    binary_dataset: list,
    ovr_records: list,
    ovo_records: dict,
):
    """
    Two-layer multiclass decoder.

    Layer 1 — OvR: each binary classifier votes; a single winner is the prediction.
    Layer 2 — OvO tiebreaker: when Layer 1 produces a tie or no winner,
               a pairwise OvO classifier resolves it.

    Parameters
    ----------
    binary_dataset : output of load_dataset_binary_balanced_with_test
    ovr_records    : list of fitted OvR binary classifiers, one per class
    ovo_records    : dict keyed by (class_a, class_b) tuples of fitted OvO classifiers

    Returns
    -------
    decoded          : final predicted class per sample
    y_test_multiclass: true multiclass labels
    """
    X_test    = binary_dataset[0][1]
    n_classes = len(binary_dataset)
    n_samples = len(X_test)

    # recover original multiclass test labels from the unmodified binary test sets
    y_test_multiclass = np.argmax(
        np.vstack([y_test for _, _, _, y_test in binary_dataset]),
        axis=0,
    )

    pred_matrix = np.empty((n_classes, n_samples))
    for i, model in enumerate(ovr_records):
        pred_matrix[i] = model.predict(X_test)

    decoded        = np.empty(n_samples)
    n_clean        = 0
    n_ovo_resolved = 0
    n_ovo_correct  = 0
    n_random_tie   = 0
    n_random_none  = 0

    for i, row in enumerate(pred_matrix.T):
        winners    = np.where(row == 1)[0]
        true_label = y_test_multiclass[i]

        if len(winners) == 1:
            decoded[i] = winners[0]
            n_clean += 1

        elif len(winners) == 2:
            pair = tuple(sorted(winners))
            if pair in ovo_records:
                ovo_pred   = ovo_records[pair].predict(X_test[[i]]).item()
                decoded[i] = pair[int(ovo_pred)]
                n_ovo_resolved += 1
                if decoded[i] == true_label:
                    n_ovo_correct += 1
            else:
                decoded[i] = random.choice(winners)
                n_random_tie += 1
                print("WARNING: no OvO model for pair", pair)

        elif len(winners) > 2:
            decoded[i] = random.choice(winners)
            n_random_tie += 1

        else:
            decoded[i] = random.choice(range(n_classes))
            n_random_none += 1

    accuracy = sum(y_test_multiclass == decoded) / n_samples
    n_bad    = n_ovo_resolved + n_random_tie + n_random_none

    print(f"{'─'*45}")
    print(f"Total samples        : {n_samples}")
    print(f"Clean OvR predictions: {n_clean}  ({n_clean/n_samples:.1%})")
    print(f"{'─'*45}")
    print(f"Ties / no winner     : {n_bad}  ({n_bad/n_samples:.1%})")
    print(f"  ├─ OvO resolved    : {n_ovo_resolved}")
    print(f"  │    └─ correct    : {n_ovo_correct}  ({n_ovo_correct/max(n_ovo_resolved,1):.1%} of OvO calls)")
    print(f"  ├─ random (tie)    : {n_random_tie}")
    print(f"  └─ random (none)   : {n_random_none}")
    print(f"{'─'*45}")
    print(f"Final accuracy       : {accuracy:.4f}")
    print(f"{'─'*45}")

    return decoded, y_test_multiclass

In [14]:
def train_ovo_tiebreakers(
    X_train: np.ndarray,
    y_train: np.ndarray,
    model_factory: str,
    config: ModelConfig,
) -> dict:
    """
    Train one pairwise OvO classifier for every combination of classes.
    These are used as the second decision layer in decode_ovr_with_ovo_tiebreaker.

    Parameters
    ----------
    X_train       : full multiclass training features
    y_train       : full multiclass training labels
    model_factory : "SVC" or "QVC"
    config        : ModelConfig (used only for QVC builds)

    Returns
    -------
    dict keyed by (class_a, class_b) tuples of fitted classifiers
    """
    unique_classes = np.unique(y_train)
    ovo_records    = {}

    for class_a, class_b in combinations(unique_classes, 2):
        mask   = np.isin(y_train, [class_a, class_b])
        X_pair = X_train[mask]
        y_pair = (y_train[mask] == class_b).astype(int)  # 0 → class_a, 1 → class_b

        pair_config = replace(config, info=f"ovo: {class_a} vs {class_b}")

        if model_factory == "QVC":
            feature_map, ansatz, optimizer, sampler = build_components(pair_config, X_train.shape[1])
            model, _, _ = train_and_save(
                sampler, feature_map, ansatz, optimizer,
                X_pair, y_pair, [], [],
                pair_config,
            )
        else:
            model = SVC()
            model.fit(X_pair, y_pair)

        ovo_records[(class_a, class_b)] = model
        print(f"OvO trained: class {class_a} vs class {class_b} | samples: {len(X_pair)}")

    return ovo_records

---
## Experiment 1 — Single multiclass VQC (baseline)

Train one VQC directly on all three Iris classes. Qiskit handles multiclass internally. This is the baseline we try to beat.

In [15]:
config = ModelConfig()
config.feature_map_fn      = "z_feature_map"
config.feature_map_rep     = 2
config.optimizer_maxiter   = 150
config.ansatz_fn           = "real_amplitudes"
config.ansatz_fn_rep       = 5
config.ansatz_fn_entanglement = "full"
config.plot_graph          = False

train_features, test_features, train_labels, test_labels, num_features = load_dataset(config)
feature_map, ansatz, optimizer, sampler = build_components(config, num_features)

vqc_multiclass, result_multiclass, record_multiclass = train_and_save(
    sampler, feature_map, ansatz, optimizer,
    train_features, train_labels,
    test_features,  test_labels,
    config,
)

Record 'a6d5e2c826ae3ece173d93a6a6805948' already exists, skipping.
Record 'a6d5e2c826ae3ece173d93a6a6805948' updated.
[DB] Total records: 64
Train: 0.6750 | Test: 0.6333 | Time: 237.3s


### Load a saved model from TinyDB

Replace `record_multiclass.id` with any stored record id to reload a previously trained model.

In [16]:
# Load a previously saved model by record id
# loaded_record = load_record(record_multiclass.id)
# model_bytes   = base64.b64decode(loaded_record.result.model_dill.encode("utf-8"))
# model_loaded  = dill.loads(model_bytes)
# print(model_loaded.score(test_features, test_labels))
# model_loaded.ansatz.draw(output="mpl", fold=20)

---
## Experiment 2 — OvR binary VQC (unbalanced)

Three binary VQCs trained with a naive OvR split. The train/test split happens before binarising labels, leaving each classifier with an imbalanced dataset (1/3 positive, 2/3 negative). Predictions are combined with naive majority vote.

In [17]:
config = ModelConfig()
config.feature_map_fn         = "z_feature_map"
config.feature_map_rep        = 1
config.optimizer_maxiter      = 100
config.ansatz_fn              = "real_amplitudes"
config.ansatz_fn_rep          = 3
config.ansatz_fn_entanglement = "full"
config.binary                 = True

binary_dataset_unbal = load_dataset_binary(config)
records_unbal        = []

for i, (tr_f, te_f, tr_l, te_l) in enumerate(binary_dataset_unbal):
    feature_map, ansatz, optimizer, sampler = build_components(config, tr_f.shape[1])
    cfg_i = replace(config, info=f"{i}")
    vqc_i, result_i, record_i = train_and_save(
        sampler, feature_map, ansatz, optimizer,
        tr_f, tr_l, te_f, te_l, cfg_i,
    )
    records_unbal.append(vqc_i)

Record '8f88c7e48ab86409806fbacedc50f7dd' already exists, skipping.
Record '8f88c7e48ab86409806fbacedc50f7dd' updated.
[DB] Total records: 64
Train: 0.8500 | Test: 0.7667 | Time: 115.8s
Record '8f1cdb50f185f1daa3311dda0600a2dd' already exists, skipping.
Record '8f1cdb50f185f1daa3311dda0600a2dd' updated.


[DB] Total records: 64
Train: 0.9667 | Test: 0.9667 | Time: 113.0s
Record '8fcbb7937469a08c93086e7f67f6a688' already exists, skipping.
Record '8fcbb7937469a08c93086e7f67f6a688' updated.
[DB] Total records: 64
Train: 0.8417 | Test: 0.7667 | Time: 114.1s


In [18]:
# Evaluate: reload the shared test set (original multiclass split)
_, X_test_unbal, _, y_test_unbal, _ = load_dataset(config)

print("=== Unbalanced OvR — naive decode ===")
_ = decode_ovr_naive(records_unbal, X_test_unbal, y_test_unbal)

=== Unbalanced OvR — naive decode ===
Bad pred ratio : 9/30 (30.0%)
Accuracy       : 0.7333


---
## Experiment 3 — OvR binary VQC (balanced)

The train/test split is performed once on the original multiclass data. Each OvR binary training set is then balanced by undersampling the negative classes equally. The test set is never resampled.

In [19]:
config = ModelConfig()
config.feature_map_fn         = "z_feature_map"
config.feature_map_rep        = 1
config.optimizer_maxiter      = 100
config.ansatz_fn              = "real_amplitudes"
config.ansatz_fn_rep          = 3
config.ansatz_fn_entanglement = "full"
config.binary                 = True

binary_dataset_bal, X_train_bal, X_test_bal, y_train_bal, y_test_bal =     load_dataset_binary_balanced_with_test(config)

records_bal = []
for i, (tr_f, te_f, tr_l, te_l) in enumerate(binary_dataset_bal):
    feature_map, ansatz, optimizer, sampler = build_components(config, tr_f.shape[1])
    cfg_i = replace(config, info=f"{i}")
    vqc_i, result_i, record_i = train_and_save(
        sampler, feature_map, ansatz, optimizer,
        tr_f, tr_l, te_f, te_l, cfg_i,
    )
    records_bal.append(vqc_i)

Class 0 vs Rest | pos: 37  neg: 36  total: 73
Class 1 vs Rest | pos: 44  neg: 44  total: 88
Class 2 vs Rest | pos: 39  neg: 38  total: 77
Record '8f88c7e48ab86409806fbacedc50f7dd' already exists, skipping.
Record '8f88c7e48ab86409806fbacedc50f7dd' updated.


[DB] Total records: 64
Train: 0.8904 | Test: 0.8667 | Time: 73.1s


Record '8f1cdb50f185f1daa3311dda0600a2dd' already exists, skipping.
Record '8f1cdb50f185f1daa3311dda0600a2dd' updated.
[DB] Total records: 64
Train: 0.9659 | Test: 0.9667 | Time: 86.7s
Record '8fcbb7937469a08c93086e7f67f6a688' already exists, skipping.
Record '8fcbb7937469a08c93086e7f67f6a688' updated.
[DB] Total records: 64
Train: 0.8961 | Test: 0.9000 | Time: 80.4s


In [43]:
print("=== Balanced OvR — naive decode ===")
_ = decode_ovr_naive(records_bal, X_test_bal, y_test_bal)

=== Balanced OvR — naive decode ===
Bad pred ratio : 4/30 (13.3%)
Accuracy       : 0.8667


---
## Experiment 4 — Classical SVC baselines

### 4a. SVC multiclass (sklearn built-in)

In [21]:
train_features, test_features, train_labels, test_labels, _ = load_dataset(config)
svc_multi = SVC()
svc_multi.fit(train_features, train_labels)
print(f"SVC multiclass — train: {svc_multi.score(train_features, train_labels):.3f} | test: {svc_multi.score(test_features, test_labels):.3f}")

SVC multiclass — train: 0.992 | test: 0.967


### 4b. SVC OvR balanced — naive decode

In [37]:
binary_dataset_svc, X_train_svc, X_test_svc, y_train_svc, y_test_svc =     load_dataset_binary_balanced_with_test(config)

records_svc = []
for i, (tr_f, te_f, tr_l, te_l) in enumerate(binary_dataset_svc):
    svc_i = SVC()
    svc_i.fit(tr_f, tr_l)
    print(f"SVC class {i} — train: {svc_i.score(tr_f, tr_l):.3f} | test: {svc_i.score(te_f, te_l):.3f}")
    records_svc.append(svc_i)

print("\n=== SVC OvR balanced — naive decode ===")
_ = decode_ovr_naive(records_svc, X_test_svc, y_test_svc)

Class 0 vs Rest | pos: 37  neg: 36  total: 73
Class 1 vs Rest | pos: 44  neg: 44  total: 88
Class 2 vs Rest | pos: 39  neg: 38  total: 77
SVC class 0 — train: 1.000 | test: 1.000
SVC class 1 — train: 1.000 | test: 0.900
SVC class 2 — train: 0.987 | test: 0.967

=== SVC OvR balanced — naive decode ===
Bad pred ratio : 4/30 (13.3%)
Accuracy       : 0.9667


### 4c. SVC OvR + OvO tiebreaker

In [23]:
ovo_records_svc = train_ovo_tiebreakers(X_train_svc, y_train_svc, model_factory="SVC", config=config)

print("\n=== SVC OvR + OvO tiebreaker ===")
decoded_svc, y_true_svc = decode_ovr_with_ovo_tiebreaker(binary_dataset_svc, records_svc, ovo_records_svc)

OvO trained: class 0 vs class 1 | samples: 81
OvO trained: class 0 vs class 2 | samples: 76
OvO trained: class 1 vs class 2 | samples: 83

=== SVC OvR + OvO tiebreaker ===
─────────────────────────────────────────────
Total samples        : 30
Clean OvR predictions: 26  (86.7%)
─────────────────────────────────────────────
Ties / no winner     : 4  (13.3%)
  ├─ OvO resolved    : 4
  │    └─ correct    : 3  (75.0% of OvO calls)
  ├─ random (tie)    : 0
  └─ random (none)   : 0
─────────────────────────────────────────────
Final accuracy       : 0.9667
─────────────────────────────────────────────


---
## Experiment 5 — QVC OvR + OvO tiebreaker (full two-layer)

The complete two-layer quantum architecture: balanced OvR VQCs as Layer 1, OvO VQC tiebreakers as Layer 2.

In [24]:
config = ModelConfig()
config.feature_map_fn         = "z_feature_map"
config.feature_map_rep        = 1
config.optimizer_maxiter      = 100
config.ansatz_fn              = "real_amplitudes"
config.ansatz_fn_rep          = 3
config.ansatz_fn_entanglement = "full"
config.binary                 = True

binary_dataset_qvc, X_train_qvc, X_test_qvc, y_train_qvc, y_test_qvc =     load_dataset_binary_balanced_with_test(config)

ovr_records_qvc = []
for i, (tr_f, te_f, tr_l, te_l) in enumerate(binary_dataset_qvc):
    feature_map, ansatz, optimizer, sampler = build_components(config, tr_f.shape[1])
    cfg_i = replace(config, info=f"{i}")
    vqc_i, _, _ = train_and_save(
        sampler, feature_map, ansatz, optimizer,
        tr_f, tr_l, te_f, te_l, cfg_i,
    )
    ovr_records_qvc.append(vqc_i)

Class 0 vs Rest | pos: 37  neg: 36  total: 73
Class 1 vs Rest | pos: 44  neg: 44  total: 88
Class 2 vs Rest | pos: 39  neg: 38  total: 77


Record '8f88c7e48ab86409806fbacedc50f7dd' already exists, skipping.
Record '8f88c7e48ab86409806fbacedc50f7dd' updated.
[DB] Total records: 64
Train: 0.8904 | Test: 0.8667 | Time: 76.2s


Record '8f1cdb50f185f1daa3311dda0600a2dd' already exists, skipping.
Record '8f1cdb50f185f1daa3311dda0600a2dd' updated.
[DB] Total records: 64
Train: 0.9659 | Test: 0.9667 | Time: 89.2s
Record '8fcbb7937469a08c93086e7f67f6a688' already exists, skipping.
Record '8fcbb7937469a08c93086e7f67f6a688' updated.
[DB] Total records: 64
Train: 0.8961 | Test: 0.9000 | Time: 78.4s


In [45]:
# OvO tiebreakers — uncomment to train QVC tiebreakers (expensive)
#ovo_records_qvc = train_ovo_tiebreakers(X_train_qvc, y_train_qvc, model_factory="QVC", config=config)

# Use SVC tiebreakers as a lighter alternative
#ovo_records_qvc = train_ovo_tiebreakers(X_train_qvc, y_train_qvc, model_factory="SVC", config=config)
ovo_records_qvc

{(np.int64(0),
  np.int64(1)): <qiskit_machine_learning.algorithms.classifiers.vqc.VQC at 0x7f23bee75520>,
 (np.int64(0),
  np.int64(2)): <qiskit_machine_learning.algorithms.classifiers.vqc.VQC at 0x7f23becada90>,
 (np.int64(1),
  np.int64(2)): <qiskit_machine_learning.algorithms.classifiers.vqc.VQC at 0x7f23bec88f80>}

In [53]:
print("\n=== QVC OvR + OvO tiebreaker ===")
decoded_qvc, y_true_qvc = decode_ovr_with_ovo_tiebreaker(binary_dataset_qvc, ovr_records_qvc, ovo_records_qvc)


=== QVC OvR + OvO tiebreaker ===
─────────────────────────────────────────────
Total samples        : 30
Clean OvR predictions: 26  (86.7%)
─────────────────────────────────────────────
Ties / no winner     : 4  (13.3%)
  ├─ OvO resolved    : 3
  │    └─ correct    : 2  (66.7% of OvO calls)
  ├─ random (tie)    : 0
  └─ random (none)   : 1
─────────────────────────────────────────────
Final accuracy       : 0.9000
─────────────────────────────────────────────
